<a href="https://colab.research.google.com/github/LovePandey/LovePandey/blob/claude%2Fexcel-analytics-tool-1BU1L/day7_multi_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.2/458.2 kB 7.0 MB/s eta 0:00:00


In [4]:
import anthropic
import pandas as pd

client = anthropic.Anthropic(api_key="")

# Reuse a similar dataset to Day 6 but add some intentional irregularities
# for the risk agent to catch
data = {
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "revenue":  [500, 520, 510, 540, 560, 575, 590, 480, 600, 625, 640, 660],  # Aug dip
    "cogs":     [300, 312, 308, 320, 330, 340, 348, 370, 355, 365, 374, 384],  # Aug COGS spike
    "opex":     [80,  82,  81,  84,  85,  86,  88,  92,  87,  89,  90,  92],
}

df = pd.DataFrame(data)
df["ebitda"] = df["revenue"] - df["cogs"] - df["opex"]
df["ebitda_margin"] = (df["ebitda"] / df["revenue"] * 100).round(1)

# Format as a clean string for the agents to consume
pl_summary = df.to_string(index=False)
print(pl_summary)

month  revenue  cogs  opex  ebitda  ebitda_margin
  Jan      500   300    80     120           24.0
  Feb      520   312    82     126           24.2
  Mar      510   308    81     121           23.7
  Apr      540   320    84     136           25.2
  May      560   330    85     145           25.9
  Jun      575   340    86     149           25.9
  Jul      590   348    88     154           26.1
  Aug      480   370    92      18            3.8
  Sep      600   355    87     158           26.3
  Oct      625   365    89     171           27.4
  Nov      640   374    90     176           27.5
  Dec      660   384    92     184           27.9


In [5]:
def run_agent(system_prompt, user_message, max_tokens=500):
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

# Agent 1: extract key metrics and trends from raw data
analyst_system = """You are a senior financial data analyst.
Your job is to analyse raw P&L data and extract the most important
metrics, trends, and observations. Be precise and data-driven.
Output structured analysis only — no waffle."""

analyst_output = run_agent(
    system_prompt=analyst_system,
    user_message=f"Analyse this P&L dataset and summarise key trends:\n\n{pl_summary}"
)

print("=== AGENT 1: DATA ANALYST ===\n")
print(analyst_output)

=== AGENT 1: DATA ANALYST ===

## P&L Analysis Summary

### Revenue Trend
- **Overall growth**: +32% (Jan: $500 → Dec: $660)
- **Average monthly growth**: ~2.5%
- **Anomaly**: August dip to $480 (18.6% below July)

### Profitability Metrics

| Metric | Jan | Dec | Change |
|--------|-----|-----|--------|
| EBITDA | $120 | $184 | +53% |
| EBITDA Margin | 24.0% | 27.9% | +390 bps |

### Critical Observations

1. **August anomaly**: EBITDA collapsed to $18 (3.8% margin) due to:
   - Revenue drop: $590 → $480 (-18.6%)
   - COGS spike: $348 → $370 (+6.3%) despite lower revenue
   - Suggests inventory write-off or one-time cost event

2. **Margin expansion**: Excluding August, consistent improvement from 24% to 28% indicates operating leverage

3. **Cost discipline**: OPEX grew only 15% ($80 → $92) against 32% revenue growth

### Flags for Further Investigation
- Root cause of August COGS anomaly
- Sustainability of margin expansion trend
- Seasonal patterns (need YoY comparison)


In [6]:
# Agent 2 receives Agent 1's output as context
risk_system = """You are a senior financial risk reviewer.
You will receive a financial analysis and your job is to identify
specific risks, red flags, and control weaknesses implied by the data.
Be direct and prioritise by severity. No generic risks — everything
must be grounded in the numbers provided."""

risk_output = run_agent(
    system_prompt=risk_system,
    user_message=f"Review this financial analysis and identify key risks:\n\n{analyst_output}"
)

print("=== AGENT 2: RISK REVIEWER ===\n")
print(risk_output)

=== AGENT 2: RISK REVIEWER ===

# Risk Assessment

## HIGH SEVERITY

### 1. August Anomaly - Potential Control Failure
The COGS spike (+6.3%) during a revenue collapse (-18.6%) is a significant red flag.

**Specific concerns:**
- COGS should move directionally with revenue; inverse movement suggests either inventory write-off being obscured, supplier contract issue or penalty, potential fraud or misclassification, or uncontrolled procurement during demand drop

**Risk:** If this was a write-off, there may be additional obsolete inventory not yet recognized. If operational, the root cause could recur.

**Required action:** Demand itemized COGS breakdown for August. Any single adjustment exceeding 5% of monthly COGS needs documentation.

---

### 2. Revenue Concentration Unknown
32% annual growth with a single-month 18.6% drop suggests potential customer or product concentration.

**Risk:** One lost contract or customer could explain August. If true, the growth narrative masks dependency

In [7]:
# Agent 3 receives both previous outputs and synthesises a final briefing
cfo_system = """You are a Chief Financial Officer writing an executive briefing.
You will receive a financial analysis and a risk review. Synthesise both into
a concise, decision-ready CFO briefing. Structure it as:
1. Headline summary (2 sentences max)
2. Key findings (3 bullets)
3. Required actions (3 bullets)
Keep it tight — this goes to the board."""

cfo_input = f"""FINANCIAL ANALYSIS:
{analyst_output}

RISK REVIEW:
{risk_output}"""

cfo_output = run_agent(
    system_prompt=cfo_system,
    user_message=f"Produce a CFO board briefing from this analysis and risk review:\n\n{cfo_input}",
    max_tokens=600
)

print("=== AGENT 3: CFO BRIEFING ===\n")
print(cfo_output)

=== AGENT 3: CFO BRIEFING ===

# CFO BOARD BRIEFING
## December Financial Review

---

### HEADLINE SUMMARY

Strong topline growth of 32% with meaningful margin expansion masks a material August anomaly that requires immediate investigation before we can validate the quality of these earnings. Until we understand the $90M COGS/revenue divergence in August, I cannot certify these results as representative of underlying performance.

---

### KEY FINDINGS

• **Growth story is real but fragile**: Revenue up 32% YoY with EBITDA margins expanding 390bps (24%→28%) driven by genuine operating leverage—OPEX grew only 15% against 32% revenue growth

• **August represents a material control gap**: COGS increased 6.3% while revenue dropped 18.6%—this inverse relationship is economically illogical and suggests either an unrecorded write-off, contract penalty, or potential misclassification requiring forensic review

• **Concentration and liquidity risks are unquantified**: A single-month 18.6% rev